# FinSight — Chunking & Retrieval Quality Analysis

This notebook explores:
- Chunk size distributions by section
- Embedding space visualization (UMAP)
- Retrieval precision vs chunk size
- BM25 vs dense vs hybrid retrieval comparison

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
# Load chunks from vectorstore metadata
from src.retrieval.bm25 import BM25Retriever
import pickle

with open('data/vectorstore/bm25_index.pkl', 'rb') as f:
    data = pickle.load(f)

chunks = data['chunks']
df = pd.DataFrame([
    {
        'chunk_id':    c['chunk_id'],
        'ticker':      c['metadata']['ticker'],
        'fiscal_year': c['metadata']['fiscal_year'],
        'section_key': c['metadata']['section_key'],
        'char_count':  len(c['text']),
        'word_count':  len(c['text'].split()),
    }
    for c in chunks
])

print(f'Total chunks: {len(df)}')
df.head()

In [ ]:
# Chunk size distribution by section
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sections = df['section_key'].unique()
for section in sections:
    sub = df[df['section_key'] == section]
    axes[0].hist(sub['char_count'], bins=30, alpha=0.6, label=section)
    axes[1].hist(sub['word_count'], bins=30, alpha=0.6, label=section)

axes[0].set_title('Character count distribution by section')
axes[0].set_xlabel('Characters per chunk')
axes[0].legend()
axes[1].set_title('Word count distribution by section')
axes[1].set_xlabel('Words per chunk')
axes[1].legend()
plt.tight_layout()
plt.show()

df.groupby('section_key')['char_count'].describe()

In [ ]:
# Chunks per company per year
pivot = df.groupby(['ticker', 'fiscal_year']).size().unstack(fill_value=0)
pivot.plot(kind='bar', figsize=(12, 5))
plt.title('Chunks indexed per company per fiscal year')
plt.ylabel('Number of chunks')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Retrieval comparison: BM25 vs Dense vs Hybrid on a test query
from src.retrieval.bm25 import BM25Retriever
from src.retrieval.vectorstore import VectorStore
from src.retrieval.hybrid import HybridRetriever

bm25   = BM25Retriever()
bm25.load()
vs     = VectorStore()
hybrid = HybridRetriever(vs, bm25, use_reranker=False)

query = 'What were Apple risk factors related to supply chain in 2023?'
filters = {'ticker': 'AAPL', 'fiscal_year': 2023}

bm25_results   = bm25.search(query, top_k=5, filters=filters)
dense_results  = vs.search(query, top_k=5, filters=filters)
hybrid_results = hybrid.retrieve(query, top_k=5, filters=filters)

print('=== BM25 top result ===')
print(bm25_results[0]['text'][:300])
print()
print('=== Dense top result ===')
print(dense_results[0]['text'][:300])
print()
print('=== Hybrid top result ===')
print(hybrid_results[0]['text'][:300])

In [ ]:
# UMAP visualization of chunk embeddings (requires umap-learn)
try:
    import umap
    
    # Get embeddings from ChromaDB
    import chromadb
    client = chromadb.PersistentClient(path='data/vectorstore')
    collection = client.get_collection('finsight_10k')
    
    # Sample 500 chunks for speed
    result = collection.get(limit=500, include=['embeddings', 'metadatas'])
    embeddings = np.array(result['embeddings'])
    metas = result['metadatas']
    
    reducer = umap.UMAP(n_components=2, random_state=42)
    reduced = reducer.fit_transform(embeddings)
    
    viz_df = pd.DataFrame(reduced, columns=['x', 'y'])
    viz_df['ticker']  = [m['ticker'] for m in metas]
    viz_df['section'] = [m['section_key'] for m in metas]
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    for ticker in viz_df['ticker'].unique():
        sub = viz_df[viz_df['ticker'] == ticker]
        axes[0].scatter(sub['x'], sub['y'], label=ticker, alpha=0.5, s=10)
    axes[0].set_title('Embedding space by company')
    axes[0].legend()
    
    for section in viz_df['section'].unique():
        sub = viz_df[viz_df['section'] == section]
        axes[1].scatter(sub['x'], sub['y'], label=section, alpha=0.5, s=10)
    axes[1].set_title('Embedding space by section')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print('Install umap-learn for embedding visualization: pip install umap-learn')